# Spike 03 — English OCR with Chandra 2 (Datalab API)

**Goal:** Extract structured English text from the Tranquil City English PDF using Chandra 2 via the Datalab REST API.

**Time box:** 1 hour

**Question to answer:** Does Chandra produce well-structured Markdown (especially tables) from this English report?

**Done when:** `ocr_output/english/tranquil_en.md` contains readable text with tables preserved as Markdown.

---

### How the API actually works (async)

```
POST /api/v1/convert  ← send the PDF
      ↓
{ "request_check_url": "https://..." }   ← get a polling URL back

GET request_check_url  (every 2 seconds)
      ↓
{ "status": "pending" }   ← not done yet, keep polling
{ "status": "complete", "markdown": "# Full extracted text..." }  ← done
```

No GPU. No RAM. No image extraction. One PDF in → Markdown out.

### API Key Setup
1. Go to [datalab.to](https://www.datalab.to) → sign up → get **\$5 free credits**
2. Add to `.env`: `CHANDRA_API_KEY=your_key_here`

## Step 1 — Imports & Setup

In [5]:
import requests
import time
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(dotenv_path="../.env")

API_KEY     = os.getenv("CHANDRA_API_KEY")
CONVERT_URL = "https://www.datalab.to/api/v1/convert"
HEADERS     = {"X-API-Key": API_KEY}   # ← X-API-Key, not Authorization: Bearer

if not API_KEY:
    raise ValueError("CHANDRA_API_KEY not found in .env file")

print("✅ Datalab client ready")
print(f"   Endpoint : {CONVERT_URL}")
print(f"   Auth     : X-API-Key: {API_KEY[:8]}...")

✅ Datalab client ready
   Endpoint : https://www.datalab.to/api/v1/convert
   Auth     : X-API-Key: kYk8JZVq...


## Step 2 — Core Functions

In [6]:
def submit_pdf(pdf_path: str, output_format: str = "markdown") -> str:
    """
    Submit a PDF to Datalab and return the check URL for polling.
    This is non-blocking — it returns immediately with a URL to check later.
    """
    with open(pdf_path, "rb") as f:
        response = requests.post(
            CONVERT_URL,
            files   = {"file": (Path(pdf_path).name, f, "application/pdf")},
            data    = {"output_format": output_format},
            headers = HEADERS,
            timeout = 30
        )

    if response.status_code != 200:
        raise RuntimeError(f"Submit failed {response.status_code}: {response.text[:300]}")

    data = response.json()
    print(f"   Submitted. Check URL: {data['request_check_url']}")
    return data["request_check_url"]


def poll_until_done(check_url: str, poll_interval: int = 2, max_wait: int = 300) -> str:
    """
    Poll the check URL every `poll_interval` seconds until status == 'complete'.
    Returns the extracted markdown text.

    max_wait: maximum seconds to wait before giving up (default 5 minutes)
    """
    waited = 0
    while waited < max_wait:
        response = requests.get(check_url, headers=HEADERS, timeout=15)
        result   = response.json()
        status   = result.get("status", "unknown")

        print(f"   Status: {status}  ({waited}s elapsed)", end="\r")

        if status == "complete":
            print(f"\n   ✅ Done in {waited}s")
            return result["markdown"]

        if status == "error":
            raise RuntimeError(f"API returned error: {result}")

        time.sleep(poll_interval)
        waited += poll_interval

    raise TimeoutError(f"Job did not complete within {max_wait}s")


def convert_pdf(pdf_path: str, output_format: str = "markdown") -> str:
    """
    Full pipeline: submit PDF → poll → return markdown.
    One call to rule them all.
    """
    print(f"Submitting: {Path(pdf_path).name}")
    check_url = submit_pdf(pdf_path, output_format)
    return poll_until_done(check_url)


print("✅ Functions defined: submit_pdf(), poll_until_done(), convert_pdf()")

✅ Functions defined: submit_pdf(), poll_until_done(), convert_pdf()


## Step 3 — Convert the English PDF

In [ ]:
import fitz  # PyMuPDF — already installed from Spike 01

OUTPUT_DIR  = Path("../ocr_output/english")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_PDF    = Path("../data/pdfs/tranquil_en.pdf")
TEMP_DIR    = Path("../data/pdfs/temp_pages")   # single-page PDFs (temp)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Same pages as Spike 02 Arabic OCR — 1-indexed filenames
SELECTED_PAGES = [7, 8, 10, 11, 20, 37, 38, 53, 60, 69, 70, 83, 107, 109]

doc = fitz.open(str(FULL_PDF))
print(f"PDF loaded: {len(doc)} total pages")
print(f"Processing: {len(SELECTED_PAGES)} selected pages\n")

for page_num in SELECTED_PAGES:
    out_md   = OUTPUT_DIR / f"page_{page_num:03d}.md"
    temp_pdf = TEMP_DIR   / f"page_{page_num:03d}.pdf"

    # Skip if already done
    if out_md.exists():
        print(f"⏭  page_{page_num:03d} — already processed")
        continue

    # Extract single page into a temp PDF
    single = fitz.open()
    single.insert_pdf(doc, from_page=page_num - 1, to_page=page_num - 1)
    single.save(str(temp_pdf))
    single.close()

    print(f"[page {page_num:03d}] Submitting ({temp_pdf.stat().st_size // 1024} KB)...", end=" ", flush=True)

    try:
        markdown = convert_pdf(str(temp_pdf))
        out_md.write_text(markdown, encoding="utf-8")
        print(f"✅ {len(markdown)} chars → {out_md.name}")
    except Exception as e:
        print(f"❌ {e}")

    temp_pdf.unlink(missing_ok=True)   # clean up temp file after each page

doc.close()
print(f"\nDone.")

## Step 4 — Inspect the Output

In [8]:
content = OUT_PATH.read_text(encoding="utf-8")

print(f"File     : {OUT_PATH.name}")
print(f"Size     : {len(content):,} characters")
print(f"Has tables (|) : {'✅ Yes' if '|' in content else '❌ No'}")
print(f"Has headings   : {'✅ Yes' if '#' in content else '❌ No'}")
print()

# Show a section from the middle of the document (more likely to have data)
mid = len(content) // 2
print("=== MIDDLE SECTION PREVIEW ===")
print(content[mid:mid + 2000])

File     : tranquil_en.md
Size     : 41,341 characters
Has tables (|) : ✅ Yes
Has headings   : ✅ Yes

=== MIDDLE SECTION PREVIEW ===
er supply and demand in Madinah from 2020 to 2026. The chart includes layers for Ground water, NTP-Approved water, Desalinated water (current), and Desalinated water (under construction). A line represents the total Demand. The chart shows a significant gap between demand and supply, which is projected to widen over time.](ecb25d766719ce041cf4cc390791a098_img.jpg)

**Madinah's Current Desalination Supply, Demand and Gap 2020 - 2026, (Million m3/d)**

| Year | Ground | NTP-Approved | Desalinated water (current) | Desalinated water (under construction) | Demand | Gap (Million m3/d) | Gap (%) |
|------|--------|--------------|-----------------------------|----------------------------------------|--------|--------------------|---------|
| 2020 | 0.18   | 0.04         | 1.04                        | 1.22                                   | 1.22   | -0.14      

## Step 5 — Spike Summary

In [ ]:
OUTPUT_DIR     = Path("../ocr_output/english")
SELECTED_PAGES = [7, 8, 10, 11, 20, 37, 38, 53, 60, 69, 70, 83, 107, 109]

md_files    = sorted(OUTPUT_DIR.glob("page_*.md"))
total_chars = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)
tables      = sum(1 for f in md_files if "|" in f.read_text(encoding="utf-8"))

print("=" * 50)
print("SPIKE 03 — RESULTS")
print("=" * 50)
print(f"OCR engine      : Chandra 2 (Datalab Cloud API)")
print(f"Pages processed : {len(md_files)} / {len(SELECTED_PAGES)}")
print(f"Total chars     : {total_chars:,}")
print(f"Avg chars/page  : {total_chars // max(len(md_files), 1):,}")
print(f"Pages with tables: {tables}")
print()
print("Files created:")
for f in md_files:
    size = len(f.read_text(encoding="utf-8"))
    print(f"  {f.name}  ({size:,} chars)")
print()

if len(md_files) == len(SELECTED_PAGES) and total_chars > 1000:
    print("✅ SPIKE PASSED — English OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
else:
    missing = [p for p in SELECTED_PAGES if not (OUTPUT_DIR / f"page_{p:03d}.md").exists()]
    print(f"⚠️  SPIKE INCOMPLETE — missing pages: {missing}")